## Libraries

pip install --upgrade "autogen-agentchat" "autogen-ext[openai]"


In [9]:
import sys
print(sys.executable)

C:\Users\Samsung\Envs\causal\Scripts\python.exe


In [1]:
#import agent_framework_ollama
import agent_framework
import pyagrum as gum
import dowhy
import matplotlib as plt
import pandas as pd
import os
import transformers
from huggingface_hub import InferenceClient
from agent_framework._clients import BaseChatClient, SupportsChatGetResponse
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI
from agent_framework._agents import Agent
from agent_framework._workflows._workflow_builder import WorkflowBuilder
from agent_framework import SessionContext
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatCompletionClient
from azure.identity import AzureCliCredential
from chirho.counterfactual.handlers import MultiWorldCounterfactual
import pyagrum.causal as csl
from funtions import pulisci_output

In [ ]:
from agents import  classificator, translator, causal_A

python-dotenv could not parse statement starting at line 13


In [3]:
from dotenv import load_dotenv
load_dotenv()

python-dotenv could not parse statement starting at line 13


True

In [9]:
dati = pd.read_csv('DATI.csv')
dati.head()

,Unnamed: 0,ILiCost,Name,Surnames,OtherCar,HomeBase,DrivHist,RiskAversion,Age,SocioEcon,...,Airbag,GoodStudent,Antilock,Mileage,SeniorTrain,DrivingSkill,Accident,RuggedAuto,ThisCarCost,VehicleYear
0,0,Thousand,Lorenzo,Liscia,True,Secure,Zero,Cautious,Adult,Middle,...,False,False,False,TwentyThou,False,Normal,NaN,Football,Thousand,Older
1,1,TenThou,Carlo,Zallio,True,Secure,Zero,Normal,Adult,Middle,...,True,False,False,FiftyThou,False,Normal,Severe,EggShell,TenThou,Older
2,2,Thousand,Caterina,Basteri,True,City,Many,Adventurous,Adult,UpperMiddle,...,False,False,False,TwentyThou,False,SubStandard,Moderate,EggShell,TenThou,Older
3,3,Thousand,Davide,Mastronardo,True,Suburb,Zero,Normal,Adult,Middle,...,True,False,True,FiftyThou,False,Normal,NaN,Football,Thousand,Current
4,4,Thousand,Luca,Uboldi,True,Suburb,Many,Normal,Senior,UpperMiddle,...,True,False,True,TwentyThou,True,SubStandard,NaN,Football,Thousand,Current


## Design

_Small Machine + Better Process_ > _Big Machine_


- Case 1:

  
      PHI-4-MINI-INSTRUCT      +      External Tool &                   >                  Nemo-3-Super                                 Multiagent Architecture

- Case 2:

       4o-mini       +      Tavily Tool &                      >                 Sonnet 4.5
                            Multiagent few-shot flow
                            
- Case 3

      

### Causal Queries

We assume that every counterfactual queries is not identifiable cause there are no assumpions on the distribution of the exogenous variables. For this reason those queries will be estimated between a range and not a precise 

In [ ]:
cond_prob = ["Given that the driving quality is normal, how likely is it that the age of the driver is senior?", "Given that the damage to a car is classified as 'Mild', how likely is that the driving quality is excellent?", "Given a situation in which the neighbourhood is labeled as 'Suburb', how likely is that the miles count is set to 'TwentyThou'?"]
counter = ['If a person’s risk aversion were ‘Cautious’ rather than ‘Psychopath’, how much would this change the likelihood of their driving being excellent?', 'If this customer were a senior citizen rather than an adult, how would the likelihood of causing a serious accident change?', 'If the customer had been wealthy rather than working-class, how would the likelihood of theft have changed?']
inter = ["How do you expect the probability of having a severe car damage if the mile age count is changed to twenty thousand?", "How do you expect the probability of having an excellent driving skill if the age is changed to senior?", "How do you expect the probability of having an older vehicleyear if the socio-economic status is set to Prole?" ]
coherence = ["Is a mild severity accident involving a car with a value of twenty thousand dollars, compatible with a cautious risk aversion profile?", "Is a drive quality value equal to normal compatible with a proposed cost equal to HundredThousand?", "Is a socio-economic status equal to Wealthy compatible with a fair cushioning system?"]

### Conversation

In [19]:
#sequential small model
async def main():
    session = causal_A.create_session()

    #task selection
    task = input()
    print(f'TASK: {task}')
    #tanslator
    translation_a = (await translator.run(f"Create a dictionary following your instruction of the query:{task}", session=session)).text
    #translation = translation_a.text
    
    #classificator
    label_a = (await classificator.run(f"classify the following query using your instructions:{task}", session = session)).text
    #label = label_a.text
    
    #causal expert
    causal_answer_a = (await causal_A.run(f"Provide the precise answer to the user's task ({task}) using your TOOLS with the follonwing parameters-> label:{label_a}, features:{translation_a}. After that give the causal explanation of this answer.", session = session)).text
    #causal_answer = causal_answer_a.text


    if "user_memory" in session.state:
        print("[Session State]", session.state["user_memory"])

    return translation_a, label_a, causal_answer_a



In [21]:
await main()

 Given that the driving quality is normal, how likely is it that the age of the driver is senior?


TASK: Given that the driving quality is normal, how likely is it that the age of the driver is senior?


('{\n  "input": {\n    "DrivQuality": "Normal"\n  },\n  "output": {\n    "Age": "P(Age=Senior | DrivQuality=Normal)"\n  }\n}',
 'backward_reasoning',
 '{\n  "function_call": {\n    "functools": [\n      {\n        "name": "read_graph",\n        "arguments": {\n          "path": "path/to/driving_quality_graph"\n        }\n      },\n      {\n        "name": "cau_inference",\n        "arguments": {\n          "label": "backward_reasoning",\n          "features": {\n            "input": {\n              "DrivQuality": "Normal"\n            },\n            "output": "P(Age=Senior | DrivQuality=Normal)"\n          },\n          "path": "path/to/driving_quality_graph/senior_driver_edge",\n          "name": "Senior Driver Age Probability"\n        }\n      }\n    ]\n  }\n}')